### Superstar – Lösung Schritt für Schritt

*37. Bundeswettbewerb Informatik (2018), 1. Runde, Aufgabe 1*

Im sozialen Netzwerk TeeniGram können Mitglieder anderen Mitgliedern „folgen“. Natürlich können sich zwei Personen gegenseitig folgen, was auch oft vorkommt.

Eine Spezialität von TeeniGram ist die Superstar-Funktion von Gruppen. Ein Mitglied X einer Gruppe heißt Superstar der Gruppe, wenn alle anderen Mitglieder der Gruppe dem Mitglied X folgen, aber X selbst keinem anderen Mitglied der Gruppe folgt.

Ein Beispiel: Eine kleine Gruppe hat drei Mitglieder, nämlich Selena, Justin und Hailey. Selena folgt Justin, Hailey folgt Justin und Hailey folgt Selena. Justin ist der Superstar.

Werbetreibende wollen sehr gerne wissen, wer in einer Gruppe der Superstar ist. In TeeniGram ist es aber nicht so einfach, an Informationen heranzukommen. Nur Anfragen der Form „Folgt Mitglied Y Mitglied Z?“ sind erlaubt, und jede Anfrage kostet einen ganzen Euro. Deshalb sollen möglichst wenige Anfragen gestellt werden.

#### Aufgabe 1

Überlege dir ein Verfahren, das nur mit Hilfe solcher Anfragen herausfindet, ob es unter den Mitgliedern einer Gruppe einen Superstar gibt. Falls die Antwort „ja“ lautet, soll der Superstar namentlich ausgegeben werden.

Versuche, die Anzahl der gestellten Anfragen klein zu halten. Kannst du die maximal notwendige Anzahl der Anfragen deines Verfahrens relativ zur Anzahl der Mitglieder der Gruppe abschätzen?

Schreibe ein Programm, das Gruppen einliest, dein Verfahren ausführt und die gestellten Anfragen samt den Antworten ausgibt. Falls ein Superstar gefunden wird, soll dieser benannt werden.

Wende dein Programm auf die Beispiele an, die du auf den BWINF-Webseiten findest.

#### Eingabeformat (`beispieldaten/superstar1.txt` bis `superstar4.txt`)

- 1. Zeile: die Namen der Gruppenmitglieder, durch Leerzeichen getrennt
- jede weitere Zeile: `X Y` bedeutet „X folgt Y“

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Ausschließen statt Suchen:** Jede Antwort auf eine Anfrage schließt ein Mitglied als Superstar aus. Was schon ausgeschlossen ist, muss man nie wieder betrachten.
- **Kandidat und Prüfung:** Erst bleibt durch Ausschließen genau ein Kandidat übrig, dann wird nur dieser eine Kandidat vollständig überprüft. Dieses Muster hilft bei vielen Aufgaben, bei denen man „höchstens einen“ Treffer sucht.
- **Warteschlange (Queue, FIFO):** Wer zuerst dran ist, wird zuerst bearbeitet. Wer übrig bleibt, wird hinten wieder angestellt.
- **Gedächtnis (Memoisierung):** Antworten, die man schon bezahlt hat, merkt man sich und fragt nicht noch einmal.

**Nützliche Funktionen und Bibliotheken**
- Mengen (`set`) und Wörterbücher (`dict`) zum schnellen Nachschlagen: `(x, y) in folgen`.
- Listen als Warteschlange: `liste.pop(0)` nimmt das erste Element heraus, `liste.append(...)` hängt hinten an.
- `random` für Zufallstests am Ende.

## Die Idee

Stell dir vor, du darfst nur fragen „Folgt `x` dem `y`?“. Was lernst du aus **einer** Antwort?

- **Antwort „ja“:** `x` folgt jemandem. Ein Superstar folgt aber niemandem. Also ist `x` **kein** Superstar. (`y` kann noch einer sein.)
- **Antwort „nein“:** `x` folgt `y` nicht. Einem Superstar müssen aber alle anderen folgen. Also ist `y` **kein** Superstar. (`x` kann noch einer sein.)

Mit jeder Anfrage scheidet also ein Mitglied aus. Fragt man immer zwei Mitglieder, die noch im Rennen sind, dann bleibt nach `n - 1` Anfragen genau **ein** Kandidat übrig (bei `n` Mitgliedern). Der Superstar, falls es einen gibt, wird nie ausgeschlossen: Er folgt niemandem und alle folgen ihm, also fällt er bei keiner Anfrage heraus.

Aber: Der Kandidat muss kein Superstar sein. Vielleicht gibt es gar keinen. Deshalb **prüfen** wir ihn danach: Folgt er jemandem, oder folgt ihm jemand nicht, dann gibt es keinen Superstar. Das kostet höchstens `2 · (n - 1)` weitere Anfragen. Insgesamt sind das höchstens `3 · (n - 1)` Anfragen, viel weniger als die `n · (n - 1)` Anfragen, die man bräuchte, wenn man einfach alle Paare fragt.

Zwei Verbesserungen sparen noch Anfragen:

1. **Antworten merken:** Bei der Prüfung kommen Paare vor, die wir schon bei der Suche gefragt haben. Sie kosten nichts mehr.
2. **Gerecht fragen (Warteschlange):** Wenn immer derselbe Kandidat gefragt wird, hat man viel über ihn gelernt, über die anderen aber wenig. Besser ist es, die Mitglieder der Reihe nach paarweise zu vergleichen: Die beiden Ersten werden verglichen, der Überlebende wird **hinten** angestellt.

## Schritt 1: Daten einlesen

Die erste Zeile enthält die Namen, jede weitere Zeile ein Paar „X folgt Y“. Die Paare speichern wir als Menge von Tupeln `(x, y)`. In einer Menge geht die Frage „Ist `(x, y)` dabei?“ besonders schnell.

In [ ]:
def lies_gruppe(dateiname):
    """Gibt (mitglieder, folgen) zurück: die Namen als Liste und die Paare (x, y) = 'x folgt y' als Menge."""
    folgen = set()
    with open(dateiname, encoding='utf-8') as datei:
        zeilen = datei.read().split('\n')
    mitglieder = zeilen[0].split()
    for zeile in zeilen[1:]:
        teile = zeile.split()
        if len(teile) == 2:
            folgen.add((teile[0], teile[1]))
    return mitglieder, folgen


for nummer in range(1, 5):
    mitglieder, folgen = lies_gruppe('beispieldaten/superstar' + str(nummer) + '.txt')
    print('superstar' + str(nummer) + ':', len(mitglieder), 'Mitglieder,', len(folgen), 'Folgen-Beziehungen, erstes Mitglied:', mitglieder[0])

superstar1: 3 Mitglieder, 3 Folgen-Beziehungen, erstes Mitglied: Selena
superstar2: 5 Mitglieder, 10 Folgen-Beziehungen, erstes Mitglied: Turing
superstar3: 8 Mitglieder, 8 Folgen-Beziehungen, erstes Mitglied: Edsger
superstar4: 80 Mitglieder, 3096 Folgen-Beziehungen, erstes Mitglied: Hanna


Das größte Beispiel hat 80 Mitglieder und 3096 Folgen-Beziehungen. Es gäbe `80 · 79 = 6320` mögliche Anfragen. Wir wollen mit viel weniger auskommen.

## Schritt 2: Die kostenpflichtige Anfrage

Das Programm darf die Menge `folgen` **nicht** einfach durchsuchen. Es darf nur über die Funktion `frage` an die Daten, und jede neue Anfrage wird mitgezählt und aufgeschrieben. Wiederholte Anfragen sind kostenlos: Die Antwort steht schon in `BEKANNT`.

Zur Ausgabe: Eine neue Anfrage bekommt eine laufende Nummer in eckigen Klammern. „Bereits bekannt“ steht bei Antworten, die wir uns gemerkt haben.

In [ ]:
def neues_netzwerk(folgen_menge):
    """Setzt Zähler und Gedächtnis zurück. Danach darf nur noch frage() benutzt werden."""
    global FOLGEN, BEKANNT, ANZAHL
    FOLGEN = folgen_menge      # die geheimen Daten des Netzwerks
    BEKANNT = {}               # (x, y) -> True oder False: schon bezahlte Antworten
    ANZAHL = 0                 # Zahl der bezahlten Anfragen


def frage(x, y, ausgeben):
    """Folgt x dem y? Kostet einen Euro, wenn wir die Antwort noch nicht kennen."""
    global ANZAHL
    if (x, y) in BEKANNT:
        if ausgeben:
            print('    Bereits bekannt:', x, 'folgt', y, '' if BEKANNT[(x, y)] else 'nicht')
        return BEKANNT[(x, y)]
    ANZAHL += 1
    antwort = (x, y) in FOLGEN
    BEKANNT[(x, y)] = antwort
    if ausgeben:
        print('[' + str(ANZAHL) + '] Beantwortete Anfrage:', x, 'folgt', y, '' if antwort else 'nicht')
    return antwort

## Schritt 3: Den Kandidaten finden

Wir stellen alle Mitglieder in eine Warteschlange (eine Liste). Solange mehr als eine Person darin steht:

1. Die ersten beiden `x` und `y` herausnehmen.
2. Fragen: Folgt `x` dem `y`?
   - **ja:** `x` ist kein Superstar, `y` kommt hinten wieder in die Schlange.
   - **nein:** `y` ist kein Superstar, `x` kommt hinten wieder in die Schlange.

Pro Durchlauf wird die Schlange um eine Person kürzer, also sind es genau `n - 1` Anfragen.

In [ ]:
def suche_kandidat(mitglieder, ausgeben):
    """Gibt das Mitglied zurück, das als einziges als Superstar übrig bleibt."""
    schlange = list(mitglieder)          # eine Kopie, die Eingabe bleibt unverändert
    while len(schlange) > 1:
        x = schlange.pop(0)              # die beiden vordersten herausnehmen
        y = schlange.pop(0)
        if frage(x, y, ausgeben):
            schlange.append(y)           # x folgt jemandem: x ist raus
        else:
            schlange.append(x)           # x folgt y nicht: y ist raus
    return schlange[0]


mitglieder2, folgen2 = lies_gruppe('beispieldaten/superstar2.txt')
neues_netzwerk(folgen2)
kandidat = suche_kandidat(mitglieder2, True)
print('Kandidat:', kandidat)

[1] Beantwortete Anfrage: Turing folgt Hoare 
[2] Beantwortete Anfrage: Dijkstra folgt Knuth nicht
[3] Beantwortete Anfrage: Codd folgt Hoare nicht
[4] Beantwortete Anfrage: Dijkstra folgt Codd nicht
Kandidat: Dijkstra


Bei `superstar2.txt` mit 5 Mitgliedern waren es genau 4 Anfragen (`n - 1`). Übrig bleibt Dijkstra. Ob er wirklich ein Superstar ist, wissen wir noch nicht.

## Schritt 4: Den Kandidaten prüfen

Für jedes andere Mitglied `x` muss gelten: `x` folgt dem Kandidaten, und der Kandidat folgt `x` nicht. Sobald eine der beiden Bedingungen verletzt ist, gibt es keinen Superstar: Alle anderen Mitglieder wurden ja schon ausgeschlossen.

Hier hilft das Gedächtnis: Hat die Suche schon nach `(kandidat, x)` oder `(x, kandidat)` gefragt, kostet es nichts.

In [ ]:
def pruefe_kandidat(kandidat, mitglieder, ausgeben):
    """True, wenn der Kandidat ein Superstar ist."""
    if ausgeben:
        print('Überprüfe', kandidat + '...')
    for x in mitglieder:
        if x != kandidat:
            if frage(kandidat, x, ausgeben):          # der Kandidat folgt jemandem
                return False
            if not frage(x, kandidat, ausgeben):      # jemand folgt dem Kandidaten nicht
                return False
    return True

## Schritt 5: Alles zusammensetzen

`superstar_suche` liest eine Datei, führt Suche und Prüfung aus und gibt die Anfragen samt Antworten aus. Bei großen Gruppen kürzen wir die Ausgabe: Mit `ausgeben = False` schreibt das Programm nur das Ergebnis.

In [ ]:
def superstar_suche(dateiname, ausgeben=True):
    mitglieder, folgen = lies_gruppe(dateiname)
    neues_netzwerk(folgen)
    kandidat = suche_kandidat(mitglieder, ausgeben)
    if pruefe_kandidat(kandidat, mitglieder, ausgeben):
        ergebnis = kandidat
        print('Superstar gefunden:', kandidat)
    else:
        ergebnis = None
        print('Kein Superstar gefunden.')
    print('Anfragen insgesamt:', ANZAHL, '(Mitglieder:', len(mitglieder), ')')
    return ergebnis


for nummer in [1, 2, 3]:
    print('--- superstar' + str(nummer) + '.txt')
    superstar_suche('beispieldaten/superstar' + str(nummer) + '.txt')

--- superstar1.txt
[1] Beantwortete Anfrage: Selena folgt Justin 
[2] Beantwortete Anfrage: Hailey folgt Justin 
Überprüfe Justin...
[3] Beantwortete Anfrage: Justin folgt Selena nicht
    Bereits bekannt: Selena folgt Justin 
[4] Beantwortete Anfrage: Justin folgt Hailey nicht
    Bereits bekannt: Hailey folgt Justin 
Superstar gefunden: Justin
Anfragen insgesamt: 4 (Mitglieder: 3 )
--- superstar2.txt
[1] Beantwortete Anfrage: Turing folgt Hoare 
[2] Beantwortete Anfrage: Dijkstra folgt Knuth nicht
[3] Beantwortete Anfrage: Codd folgt Hoare nicht
[4] Beantwortete Anfrage: Dijkstra folgt Codd nicht
Überprüfe Dijkstra...
[5] Beantwortete Anfrage: Dijkstra folgt Turing nicht
[6] Beantwortete Anfrage: Turing folgt Dijkstra 
[7] Beantwortete Anfrage: Dijkstra folgt Hoare nicht
[8] Beantwortete Anfrage: Hoare folgt Dijkstra 
    Bereits bekannt: Dijkstra folgt Knuth nicht
[9] Beantwortete Anfrage: Knuth folgt Dijkstra 
    Bereits bekannt: Dijkstra folgt Codd nicht
[10] Beantwortete Anfrage

Das sind genau die Abläufe der Lösungshinweise (dort für `superstar1.txt` ebenfalls 4, für `superstar2.txt` 10 und für `superstar3.txt` 8 Anfragen). Bei `superstar3.txt` ist schon bei der ersten Prüffrage klar, dass der Kandidat Jitse jemandem folgt (Edsger): Es gibt keinen Superstar.

In [ ]:
print('--- superstar4.txt (nur das Ergebnis)')
ergebnis4 = superstar_suche('beispieldaten/superstar4.txt', False)

--- superstar4.txt (nur das Ergebnis)
Superstar gefunden: Folke
Anfragen insgesamt: 230 (Mitglieder: 80 )


Beim großen Beispiel mit 80 Mitgliedern ist **Folke** der Superstar, gefunden mit 230 Anfragen statt 6320 möglichen. Zum Vergleich die Schranke `3 · (n - 1) = 237` aus der Idee.

## Wie viele Anfragen braucht das Verfahren höchstens?

- **Suche:** genau `n - 1` Anfragen.
- **Prüfung:** für jedes der `n - 1` anderen Mitglieder höchstens 2 Anfragen, also höchstens `2 · (n - 1)`.
- **Zusammen:** höchstens **`3 · (n - 1)`** Anfragen, also im Wesentlichen *linear* in der Zahl der Mitglieder. Alle Paare zu fragen kostet dagegen `n · (n - 1)` und wächst quadratisch.

Beispiel: Bei 1000 Mitgliedern braucht unser Verfahren höchstens 2997 Anfragen, die Alle-Paare-Methode 999 000.

Die Antworten der Suche zählen bei der Prüfung mit, deshalb bleiben wir oft darunter (230 statt 237). Wir prüfen die Schranke mit Zufallsgruppen: einmal mit Superstar, einmal ohne. Dazu bauen wir die Folgen-Menge selbst.

In [ ]:
import random


def zufallsgruppe(n, mit_superstar, wahrscheinlichkeit, zufall):
    """n Mitglieder 'm0' ...; jeder folgt jedem anderen mit der angegebenen Wahrscheinlichkeit.
    Mit Superstar: ein zufälliges Mitglied folgt niemandem und alle anderen folgen ihm."""
    mitglieder = ['m' + str(i) for i in range(n)]
    folgen = set()
    for x in mitglieder:
        for y in mitglieder:
            if x != y and zufall.random() < wahrscheinlichkeit:
                folgen.add((x, y))
    if mit_superstar:
        stern = mitglieder[zufall.randrange(n)]
        folgen = {(x, y) for (x, y) in folgen if x != stern}
        for x in mitglieder:
            if x != stern:
                folgen.add((x, stern))
    return mitglieder, folgen


zufall = random.Random(2018)
schlimmster_quotient = 0
fehler = 0
for versuch in range(300):
    n = zufall.randint(2, 40)
    mit = versuch % 2 == 0
    mitglieder, folgen = zufallsgruppe(n, mit, zufall.choice([0.1, 0.5, 0.9]), zufall)
    neues_netzwerk(folgen)
    kandidat = suche_kandidat(mitglieder, False)
    gefunden = pruefe_kandidat(kandidat, mitglieder, False)
    wahr = False                                  # zur Kontrolle: alle Paare ansehen (nur im Test erlaubt)
    for s in mitglieder:
        if all((x, s) in folgen and (s, x) not in folgen for x in mitglieder if x != s):
            wahr = True
    if gefunden != wahr:
        fehler += 1                               # das Verfahren hätte sich geirrt
    schlimmster_quotient = max(schlimmster_quotient, ANZAHL / (3 * (n - 1)))
print('Fehler:', fehler)
print('Größte Anfragenzahl im Verhältnis zu 3(n-1):', round(schlimmster_quotient, 2))

Fehler: 0
Größte Anfragenzahl im Verhältnis zu 3(n-1): 0.96


Bei 300 Zufallsgruppen (2 bis 40 Mitglieder, davon die Hälfte mit eingebautem Superstar) stimmte das Ergebnis jedes Mal mit einer Kontrolle über alle Paare überein, und die Schranke `3 · (n - 1)` wurde nie überschritten (der Quotient bleibt höchstens 1).

## Für Neugierige: Wie viel bringen die beiden Verbesserungen?

Wir vergleichen drei Varianten bei `superstar4.txt`:

- **A:** Suche mit einem „laufenden Kandidaten“ (er wird nacheinander mit allen anderen verglichen), Prüfung ohne Gedächtnis: `(n - 1)` Fragen Suche plus zwei Fragen pro anderem Mitglied.
- **B:** wie A, aber mit Gedächtnis.
- **C:** unser Verfahren mit Warteschlange und Gedächtnis.

In [ ]:
def suche_kandidat_laufend(mitglieder):
    """Der Kandidat wird der Reihe nach mit allen anderen verglichen."""
    kandidat = mitglieder[0]
    for y in mitglieder[1:]:
        if frage(kandidat, y, False):
            kandidat = y             # der bisherige Kandidat folgt jemandem
    return kandidat


mitglieder4, folgen4 = lies_gruppe('beispieldaten/superstar4.txt')
n4 = len(mitglieder4)
print('A: laufender Kandidat, ohne Gedächtnis:', (n4 - 1) + 2 * (n4 - 1), 'Anfragen')

neues_netzwerk(folgen4)
kandidat = suche_kandidat_laufend(mitglieder4)
pruefe_kandidat(kandidat, mitglieder4, False)
anzahl_b = ANZAHL
print('B: laufender Kandidat, mit Gedächtnis: ', anzahl_b, 'Anfragen')

neues_netzwerk(folgen4)
kandidat = suche_kandidat(mitglieder4, False)
pruefe_kandidat(kandidat, mitglieder4, False)
anzahl_c = ANZAHL
print('C: Warteschlange, mit Gedächtnis:      ', anzahl_c, 'Anfragen')

A: laufender Kandidat, ohne Gedächtnis: 237 Anfragen
B: laufender Kandidat, mit Gedächtnis:  182 Anfragen
C: Warteschlange, mit Gedächtnis:       230 Anfragen


Das Gedächtnis bringt viel (A: 237, B: 182 Anfragen). Überraschend ist, dass die Warteschlange (C: 230) bei diesem Beispiel **schlechter** ist als der laufende Kandidat (B: 182). Das liegt an den Daten: Der laufende Kandidat wird gegen alle verglichen, und beim Superstar Folke ist am Ende fast alles schon bekannt, sobald er der Kandidat ist. Bei der Warteschlange kommt Folke erst spät in die Runde, und viele Prüffragen sind neu. Die Lösungshinweise empfehlen die Warteschlange, weil sie nicht von der Reihenfolge der Namen abhängt: Steht der Superstar ganz am Ende der Liste, wird er vom laufenden Kandidaten nur ein einziges Mal gefragt, und die Prüfung braucht fast alle `2 · (n - 1)` Fragen, also fast `3 · (n - 1)` insgesamt. Steht er dagegen ganz vorne, ist der laufende Kandidat sogar besser. Im schlechtesten Fall bleiben beide Verfahren bei höchstens `3 · (n - 1)`; welches im Einzelfall weniger Anfragen braucht, hängt von der Reihenfolge der Namen ab.

## Zum Schluss: Superstar mit MiniZinc?

Wir haben geprüft, ob sich die Aufgabe mit MiniZinc sinnvoll lösen lässt, und ein einfaches Modell mit allen vier Beispielen gemessen: „Finde `s`, sodass alle anderen `s` folgen und `s` niemandem folgt.“ Der Solver (Gecode) antwortet bei allen vier Beispielen in rund 0,2 Sekunden und findet dieselben Ergebnisse wie unser Programm: Justin, Dijkstra, kein Superstar, Folke.

Trotzdem haben wir kein Modell aufgenommen:

- Das Modell braucht die **ganze** Tabelle „wer folgt wem“ als Eingabe. Gerade das verbietet die Aufgabe: Jede einzelne Antwort kostet einen Euro.
- Was die Aufgabe eigentlich verlangt, ist ein **Frageverfahren**, das aus den bisherigen Antworten die nächste Frage bestimmt. Das ist eine Strategie und kein Zuordnungs- oder Optimierungsproblem; MiniZinc beschreibt aber nur, wie eine fertige Lösung aussieht.
- Mit der Tabelle in der Hand ist die Antwort auch ohne Solver in einem Durchlauf zu bekommen.

Der Algorithmus oben ist hier die richtige Lösung.